# Определение возраста покупателей

## Исследовательский анализ данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Загрузка данных
df = pd.read_csv('/datasets/faces/labels.csv')

# Размер выборки и первые строки
print('Размер выборки:', df.shape)
display(df.head())

In [ ]:
# Информация о датафрейме
df.info()

In [ ]:
# Гистограмма и диаграмма размаха распределения возраста
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [3, 1]})

# Гистограмма с bins=100
ax1.hist(df['real_age'], bins=100, color='steelblue', edgecolor='white')
ax1.set_xlabel('Возраст')
ax1.set_ylabel('Количество')
ax1.set_title('Распределение возраста')

# Boxplot с той же осью Y
ax2.boxplot(df['real_age'], vert=True)
ax2.set_ylabel('Возраст')
ax2.set_title('Диаграмма размаха')
ax2.set_ylim(ax1.get_ylim())

plt.tight_layout()
plt.show()

print(f"Минимальный возраст: {df['real_age'].min()}")
print(f"Максимальный возраст: {df['real_age'].max()}")
print(f"Средний возраст: {df['real_age'].mean():.1f}")
print(f"Медиана: {df['real_age'].median()}")

In [ ]:
# Просмотр примеров изображений
datagen = ImageDataGenerator(rescale=1/255)
gen = datagen.flow_from_dataframe(
    dataframe=df,
    directory='/datasets/faces/final_files/',
    x_col='file_name',
    y_col='real_age',
    target_size=(150, 150),
    batch_size=15,
    class_mode='raw',
    seed=42
)

images, labels = next(gen)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i])
    ax.set_title(f'Возраст: {int(labels[i])}')
    ax.axis('off')
plt.suptitle('Примеры изображений из датасета', fontsize=14)
plt.tight_layout()
plt.show()

<div class="alert" style="background-color:#ead7f7;color:#8737bf">
    <font size="3"><b>Комментарий студента — Выводы по EDA</b></font>

**Размер датасета:**
Датасет содержит **7591 изображение** лиц с размеченным возрастом. Это умеренный объём для задачи регрессии; с применением transfer learning и аугментаций должно быть достаточно для достижения MAE < 8.

**Распределение целевого признака (возраст):**
- Минимальный возраст — около 1 года, максимальный — около 100 лет.
- Большая часть фотографий сконцентрирована в диапазоне 15–40 лет.
- На гистограмме с bins=100 заметны всплески в «юбилейные» возрасты (20, 25, 30, 35 лет и т.д.), что может указывать на неточную разметку — люди склонны округлять возраст при заполнении данных.
- Данные несбалансированы: детей и пожилых людей значительно меньше, чем молодых взрослых.

**Особенности изображений (по визуальному осмотру):**
- На фото преимущественно лица крупным планом — это хорошо для нашей задачи.
- Встречаются чёрно-белые фотографии — модель должна работать с обоими форматами.
- Присутствуют различные позы и повороты головы.
- На некоторых фото есть посторонние предметы и частично закрытые лица.

**Рекомендуемые аугментации:**
- Горизонтальное отражение (flip) — разнообразит позы
- Случайный поворот (rotation) — учтёт наклоны головы
- Небольшое изменение яркости (brightness) — компенсирует разные условия освещения
- Обрезка (zoom/crop) — имитирует разное расстояние до камеры
</div>

## Обучение модели

```python
import numpy as np
import pandas as pd
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


def load_train(path):
    df = pd.read_csv('/datasets/faces/labels.csv')
    datagen = ImageDataGenerator(
        rescale=1/255,
        horizontal_flip=True,
        rotation_range=20,
        brightness_range=(0.8, 1.2),
        zoom_range=0.1,
        validation_split=0.2
    )
    train_gen = datagen.flow_from_dataframe(
        dataframe=df,
        directory=path,
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='training',
        seed=42
    )
    return train_gen


def load_test(path):
    df = pd.read_csv('/datasets/faces/labels.csv')
    datagen = ImageDataGenerator(rescale=1/255, validation_split=0.2)
    test_gen = datagen.flow_from_dataframe(
        dataframe=df,
        directory=path,
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='validation',
        seed=42
    )
    return test_gen


def create_model(input_shape):
    backbone = ResNet50(
        input_shape=input_shape,
        weights='imagenet',
        include_top=False
    )
    backbone.trainable = True

    model = Sequential([
        backbone,
        GlobalAveragePooling2D(),
        Dense(1, activation='relu')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='mse',
        metrics=['mae']
    )
    return model


def train_model(model, train_data, test_data, batch_size=32, epochs=10, steps_per_epoch=None, validation_steps=None):
    if steps_per_epoch is None:
        steps_per_epoch = len(train_data)
    if validation_steps is None:
        validation_steps = len(test_data)

    history = model.fit(
        train_data,
        validation_data=test_data,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=2
    )
    return history


def main():
    path = '/datasets/faces/final_files/'
    train_data = load_train(path)
    test_data = load_test(path)
    model = create_model((224, 224, 3))
    model.summary()
    train_model(model, train_data, test_data, epochs=10)


main()
```

```
Epoch 1/10
190/190 - 57s - loss: 209.0138 - mae: 10.4112 - val_loss: 706.3535 - val_mae: 21.4739
Epoch 2/10
190/190 - 38s - loss: 76.1480 - mae: 6.5946 - val_loss: 573.9771 - val_mae: 18.7442
Epoch 3/10
190/190 - 38s - loss: 50.8692 - mae: 5.4708 - val_loss: 262.2289 - val_mae: 11.7833
Epoch 4/10
190/190 - 38s - loss: 35.2431 - mae: 4.5601 - val_loss: 116.3549 - val_mae: 8.4322
Epoch 5/10
190/190 - 38s - loss: 27.1485 - mae: 4.0034 - val_loss: 71.6150 - val_mae: 6.3774
Epoch 6/10
190/190 - 39s - loss: 21.8034 - mae: 3.6045 - val_loss: 86.8629 - val_mae: 7.1660
Epoch 7/10
190/190 - 38s - loss: 20.0089 - mae: 3.4316 - val_loss: 70.5638 - val_mae: 6.2290
Epoch 8/10
190/190 - 39s - loss: 15.4143 - mae: 3.0463 - val_loss: 79.8217 - val_mae: 7.0984
Epoch 9/10
190/190 - 43s - loss: 14.0216 - mae: 2.8045 - val_loss: 72.6768 - val_mae: 6.5997
Epoch 10/10
190/190 - 47s - loss: 11.2989 - mae: 2.5538 - val_loss: 67.4160 - val_mae: 6.0481
```

## Анализ обученной модели

<div class="alert" style="background-color:#ead7f7;color:#8737bf">
    <font size="3"><b>Комментарий студента — Анализ обученной модели</b></font>

**Архитектура модели:**
Использована предобученная сеть **ResNet50** (веса ImageNet) с размороженными слоями для дообучения (fine-tuning). Поверх backbone добавлены:
- `GlobalAveragePooling2D` — агрегирует карты признаков
- `Dense(1, activation='relu')` — выходной нейрон для предсказания возраста (relu гарантирует неотрицательное значение)

**Параметры обучения:**
- Loss: MSE (Mean Squared Error)
- Метрика: MAE (Mean Absolute Error)
- Оптимизатор: Adam (lr=0.0001)
- Эпох: 10, batch_size: 32

**Результаты обучения:**
- **MAE на обучающей выборке (epoch 10): 2.55**
- **MAE на валидационной выборке (epoch 10): 6.05**
- Требование проекта (MAE ≤ 8) — **выполнено** ✅

**Наблюдения:**
- Модель демонстрирует хорошее итоговое качество, но с выраженным **переобучением**: train_MAE = 2.55 против val_MAE = 6.05, разрыв составляет ~3.5. Это говорит о том, что модель слишком хорошо «запомнила» обучающую выборку.
- Заметна **нестабильность на валидации** в середине обучения — val_MAE колебалась между 6.2 и 7.2 на эпохах 5–9, тогда как на обучающей выборке метрика монотонно улучшалась. Это типичное поведение при сильном переобучении.
- При необходимости улучшить результат можно добавить слой `Dropout` перед выходным Dense, усилить аугментации или применить `EarlyStopping` по val_mae.

**Вывод:**
Модель на основе ResNet50 с transfer learning успешно справляется с задачей определения возраста по фотографии. Итоговый val_MAE ≈ 6.05 означает, что в среднем модель ошибается в определении возраста примерно на 6 лет, что является хорошим результатом для данной задачи.
</div>

## Чек-лист

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Исследовательский анализ данных выполнен
- [x]  Результаты исследовательского анализа данных перенесены в финальную тетрадь
- [x]  MAE модели не больше 8
- [x]  Код обучения модели скопирован в финальную тетрадь
- [x]  Результат вывода модели на экран перенесён в финальную тетрадь
- [x]  По итогам обучения модели сделаны выводы